# Kepler Light Curve — Trend, Variability & Uncertainty

**DS-III Data Visualization — Deployed Streamlit App + Notebook**

- **Dataset:** KIC 11395018, Kepler long-cadence stitched light curve
  (FITS: `hlsp_kepler.fits`, extension `LIGHTCURVE_STITCHED`, from MAST).
- **Companion app:** the deployed Streamlit version lives at
  `https://q6dulz3jf99abp9evx2ceq.streamlit.app/` and the source is in the same repo.
- **Goal:** separate long-term trend from short-term variability via STL,
  show uncertainty honestly, and demonstrate how encoding choices affect
  what the same data communicates.

## 1. Environment

```bash
python -m venv .venv
.venv\Scripts\activate          # Windows
pip install pandas numpy matplotlib altair statsmodels
```

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import altair as alt
from statsmodels.tsa.seasonal import STL

alt.data_transformers.disable_max_rows()

print('pandas   ', pd.__version__)
print('numpy    ', np.__version__)
print('altair   ', alt.__version__)

## 2. Load the dataset

The CSV was derived from the stitched FITS file with `astropy.io.fits`.
`time` is BJD − 2454833 (days since the Kepler launch epoch).

In [ ]:
HERE = Path.cwd()
DATA_FILE = HERE / 'kic11395018_lightcurve.csv'
print('Looking for:', DATA_FILE)
print('Exists:', DATA_FILE.exists())

df = pd.read_csv(DATA_FILE).sort_values('time').reset_index(drop=True)

# BJD -> real datetime
epoch = pd.Timestamp('2009-05-02')
df['datetime'] = epoch + pd.to_timedelta(df['time'], unit='D')

# Relative flux centered near 1.0
median_flux = df['flux'].median()
df['rel_flux'] = df['flux'] / median_flux
if 'flux_err' in df.columns:
    df['rel_flux_err'] = df['flux_err'] / median_flux
else:
    df['rel_flux_err'] = 0.0

# Quality flag
if 'sap_quality' in df.columns:
    df['sap_quality'] = df['sap_quality'].fillna(0).astype(int)
else:
    df['sap_quality'] = 0

print(f'Rows: {len(df):,}')
df.head()

## 3. Data preview

In [ ]:
df.describe()

## 4. Resample to daily bins

Kepler long-cadence integrates 29.4 minutes per point. Binning to daily
averages makes the STL decomposition tractable and honest — the cadence
is disclosed in the caption.

In [ ]:
plot_df = (
    df.set_index('datetime')['rel_flux']
      .resample('1D').mean()
      .dropna()
      .reset_index()
      .rename(columns={'datetime': 'date'})
)
print(f'Daily rows: {len(plot_df):,}')
plot_df.head()

## 5. Rolling uncertainty band

The band combines pipeline photon noise (`flux_err`) with a rolling
standard deviation (systematics). A fixed global band would misrepresent
heteroskedasticity across quarters.

In [ ]:
window = 7
roll_mean = plot_df['rel_flux'].rolling(window, center=True, min_periods=1).mean()
roll_std  = plot_df['rel_flux'].rolling(window, center=True, min_periods=1).std()

# align rel_flux_err by date
err_series = (
    df.set_index('datetime')['rel_flux_err']
      .resample('1D').mean()
      .reindex(plot_df['date'])
      .fillna(0)
      .values
)

total_sigma = np.sqrt(roll_std.fillna(0).values ** 2 + err_series ** 2)

band_df = pd.DataFrame({
    'date':  plot_df['date'],
    'mean':  roll_mean.values,
    'lower': roll_mean.values - 1.96 * total_sigma,
    'upper': roll_mean.values + 1.96 * total_sigma,
})
band_df.head()

## 6. Main chart — Light curve with uncertainty

Static Matplotlib version for the report.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))

ax.fill_between(band_df['date'], band_df['lower'], band_df['upper'],
                color='#4c78a8', alpha=0.25, label='95% rolling band')
ax.plot(band_df['date'], band_df['mean'], color='#1f4e79', lw=1.5,
        label='Rolling mean')
ax.scatter(plot_df['date'], plot_df['rel_flux'], s=4, color='#555',
           alpha=0.3, label='Daily averages')

ax.set_xlabel('Date')
ax.set_ylabel('Relative flux')
ax.set_title('KIC 11395018 — relative flux with rolling 95% band')
ax.grid(alpha=0.3)
ax.legend(loc='lower left', fontsize=9)
plt.tight_layout()
plt.show()

Interactive Altair version of the same chart.

In [ ]:
band = alt.Chart(band_df).mark_area(opacity=0.25, color='#4c78a8').encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('lower:Q', title='Relative flux', scale=alt.Scale(zero=False)),
    y2='upper:Q',
)
line = alt.Chart(band_df).mark_line(color='#1f4e79', strokeWidth=1.5).encode(
    x='date:T', y='mean:Q'
)
points = (
    alt.Chart(plot_df)
    .mark_circle(size=5, opacity=0.25, color='#555')
    .encode(
        x='date:T', y='rel_flux:Q',
        tooltip=['date:T', alt.Tooltip('rel_flux:Q', format='.5f')],
    )
)
(band + line + points).properties(height=380).interactive()

## 7. STL decomposition

Trend + seasonal (7-day period) + residual.

In [ ]:
daily = (
    df.set_index('datetime')['rel_flux']
      .resample('1D').mean().dropna()
)
res = STL(daily, period=7, robust=True).fit()

stl_df = pd.DataFrame({
    'date':     daily.index,
    'observed': daily.values,
    'trend':    res.trend.values,
    'seasonal': res.seasonal.values,
    'resid':    res.resid.values,
})

fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
axes[0].plot(stl_df['date'], stl_df['observed'], color='#333', lw=0.7)
axes[0].set_ylabel('Observed')
axes[0].set_title('STL Decomposition — KIC 11395018', fontsize=13)
axes[1].plot(stl_df['date'], stl_df['trend'], color='#d62728', lw=1.5)
axes[1].set_ylabel('Trend')
axes[2].plot(stl_df['date'], stl_df['seasonal'], color='#2ca02c', lw=0.9)
axes[2].set_ylabel('Seasonal (7-day)')
axes[3].plot(stl_df['date'], stl_df['resid'], color='#7f7f7f', lw=0.7)
axes[3].set_ylabel('Residual')
axes[3].set_xlabel('Date')
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Faceted Altair version — same data, three panels side by side.

In [ ]:
stl_long = pd.DataFrame({
    'date': list(stl_df['date']) * 3,
    'value': np.concatenate([stl_df['trend'].values,
                             stl_df['seasonal'].values,
                             stl_df['resid'].values]),
    'component': (['Trend'] * len(stl_df) +
                  ['Seasonal (7-day)'] * len(stl_df) +
                  ['Residual'] * len(stl_df)),
})

colors = alt.Scale(
    domain=['Trend', 'Seasonal (7-day)', 'Residual'],
    range=['#d62728', '#2ca02c', '#7f7f7f'],
)

(alt.Chart(stl_long)
  .mark_line()
  .encode(
      x=alt.X('date:T', title='Date'),
      y=alt.Y('value:Q', title='Component value'),
      color=alt.Color('component:N', title='Component', scale=colors),
      tooltip=['date:T', 'component:N', alt.Tooltip('value:Q', format='.5f')],
  )
  .properties(height=140)
  .facet(row=alt.Row('component:N', title=None,
                     header=alt.Header(labelAngle=0, labelAlign='left')))
  .resolve_scale(y='independent'))

## 8. Encoding playground

The same data can say different things depending on which variable you put
on which visual channel. Position is read most accurately; area least.

In [ ]:
# Fixed channel assignments — mirror the Streamlit playground here
X_FIELD     = 'time'
Y_FIELD     = 'rel_flux'
SIZE_FIELD  = '(none)'      # or 'flux', 'sap_flux', 'rel_flux_err', ...
COLOR_FIELD = 'quarter'     # or '(none)' or any numeric field

playground_src = df.copy()
playground_src['quarter'] = playground_src['quarter'].astype('Int64').astype(str)

if len(playground_src) > 5000:
    playground_src = playground_src.sample(5000, random_state=0)

enc = {
    'x': alt.X(f'{X_FIELD}:Q', title=X_FIELD),
    'y': alt.Y(f'{Y_FIELD}:Q', title=Y_FIELD),
    'tooltip': [
        alt.Tooltip('datetime:T', title='Date'),
        alt.Tooltip(f'{X_FIELD}:Q', title=X_FIELD, format='.4f'),
        alt.Tooltip(f'{Y_FIELD}:Q', title=Y_FIELD, format='.4f'),
    ],
}

if SIZE_FIELD != '(none)':
    enc['size'] = alt.Size(f'{SIZE_FIELD}:Q', title=SIZE_FIELD,
                           scale=alt.Scale(range=[10, 400]))
else:
    enc['size'] = alt.value(30)

if COLOR_FIELD == '(none)':
    enc['color'] = alt.value('#4c78a8')
elif COLOR_FIELD == 'quarter':
    enc['color'] = alt.Color('quarter:N', title='quarter')
else:
    enc['color'] = alt.Color(f'{COLOR_FIELD}:Q', title=COLOR_FIELD,
                             scale=alt.Scale(scheme='viridis'))

(alt.Chart(playground_src)
  .mark_circle(opacity=0.55)
  .encode(**enc)
  .properties(height=420)
  .interactive())

### Perceptual justification

- **Position on a common scale** is the most accurate quantitative channel
  (Cleveland & McGill, 1984; Munzner, 2014).
- **Size / area** is far less accurate — the eye underestimates large
  circles and overestimates small ones. Use it for ordinal signals only.
- **Hue** is categorical — good for identity, poor for ordering. Use
  sequential scales (viridis here) for ordinal or continuous variables.
- **Tooltips** surface exact values on hover, avoiding the need to
  decode size or hue precisely.

## 9. Temporal-honesty note

**Cadence disclosure.** Kepler long-cadence integrates 29.4 minutes per
measurement. Binning to daily averages suppresses real short-period
variability; the caption states the active resolution.

**Quarter gaps.** The x-axis uses real datetimes, so the ~1-day downlinks
between Kepler quarters appear as horizontal gaps. This is truthful —
the telescope was not observing during those intervals.

**Quality flags.** Non-zero `sap_quality` cadences are retained by default
so the reader sees the raw picture first. The companion Streamlit app
exposes a toggle; the choice is explicit, never silent.

**Axis honesty.** The y-axis uses `scale(zero=False)` because relative
flux spans a narrow range (~0.99–1.01). Forcing zero would compress the
signal into a flat line. Uncertainty combines pipeline photon noise with
rolling systematics; a fixed global band would misrepresent
heteroskedasticity across quarters.

## 10. Companion Streamlit app

The deployed interactive version lives at:
`https://q6dulz3jf99abp9evx2ceq.streamlit.app/`

Source: `app.py` in the same repository. It provides:

- Sidebar widgets: resolution radio, quality-flag checkbox, quarter multiselect
- Six tabs: Overview, Light Curve, STL, Encodings, Data, About
- Caching via `@st.cache_data` on loading and STL
- Download button for the filtered dataset